In [ ]:
#Opgave 1.a
import pandas as pd

df = pd.read_stata('Integration.dta')

df.head()

dummies = pd.get_dummies(df['ankomstaar'].astype(int), prefix='ankomst')
df = pd.concat([df, dummies], axis=1)
kolonner_til_tabel = ['aarslon', 'alder', 'ankomst_2014', 'ankomst_2015', 'ankomst_2016']
deskriptiv_tabel = df.groupby(['mand', 'aar'])[kolonner_til_tabel].mean().round(2)
deskriptiv_tabel['N'] = df.groupby(['mand', 'aar']).size()

display(deskriptiv_tabel)

aarslon      alder  ankomst_2014  ankomst_2015  \
mand aar                                                           
0.0  2017.0  31124.439453  28.629999          0.34          0.34   
     2018.0  40045.050781  29.629999          0.34          0.34   
1.0  2017.0  61324.039062  28.670000          0.33          0.32   
     2018.0  90906.500000  29.670000          0.33          0.32   

             ankomst_2016     N  
mand aar                         
0.0  2017.0          0.32   495  
     2018.0          0.32   495  
1.0  2017.0          0.35  1905  
     2018.0          0.35  1905

In [24]:
#Opgave 1.b
import statsmodels.api as sm
import numpy as np
import scipy.stats as stats

df['YSM'] = df['aar']- df['ankomstaar']
df['ln_aarslon']= np.log(df['aarslon'])
df_2017 = df[df['aar'] == 2017].copy()

df_2017['const'] = 1

y = df_2017['ln_aarslon'].values
X = df_2017[['const', 'alder', 'YSM']].values

navne = ['Konstant', 'Alder', 'YSM']
def OLS_ren(X, y, navne):

    n = X.shape[0]
    k = X.shape[1] - 1


    X_T_X_inv = np.linalg.inv(X.T @ X)
    betahat = X_T_X_inv @ X.T @ y
    
    y_hat = X @ betahat
    u_hat = y - y_hat
    y_bar = np.mean(y)

    SST = np.sum((y - y_bar)**2)
    SSR = np.sum(u_hat**2)
    SSE = np.sum((y_hat - y_bar)**2)
    R2 = SSE / SST
    
    resvar = (1 / (n - k - 1)) * np.dot(u_hat, u_hat)
    
    var_betahatX = resvar * X_T_X_inv
    se_betahat = np.sqrt(np.diagonal(var_betahatX))
    
    t_stat = betahat / se_betahat
    p_values = 2 * stats.t.sf(np.abs(t_stat), n - k - 1)

    print("==============================================================================")
    print(f"Number of Observations: {n}")
    print(f"Residual Degrees of Freedom: {n - k - 1}")
    print(f"R-squared: {R2:.4f}")
    print(f"SST: {SST:.4f}, SSE: {SSE:.4f}, SSR: {SSR:.4f}")
    print("==============================================================================")
    
    #tabel layout
    results = pd.DataFrame({
        'Koefficient': betahat,
        'Std.fejl': se_betahat,
        't-værdi': t_stat,
        'P>|t|': p_values
    }, index=navne)
    
    formatted_results = results.copy()
    for col in formatted_results.columns:
        formatted_results[col] = formatted_results[col].apply(lambda x: f"{x:.4f}")
    
    display(formatted_results)

# Kald funktionen med dine definerede variabler
OLS_ren(X, y, navne)

Number of Observations: 2400
Residual Degrees of Freedom: 2397
R-squared: 0.0322
SST: 569.3583, SSE: 18.3175, SSR: 551.0407


,Koefficient,Std.fejl,t-værdi,P>|t|
Konstant,10.4231,0.0474,219.7993,0.0000
Alder,0.0071,0.0014,5.1251,0.0000
YSM,0.0870,0.0119,7.3310,0.0000
